#### Exercise 1: CSVReader Class

A reusable class for loading a CSV file with pandas and previewing its top rows.<br><br>
Objective:<br>
•	Reinforce OOP fundamentals: classes, constructors, methods, encapsulation.<br>
•	Learn to design reusable data classes for DS projects.

In [7]:
import pandas as pd


class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None

    def read(self):
        """Load the CSV file and return it as a pandas DataFrame."""
        self._data = pd.read_csv(self.file_path)
        return self._data

    def preview(self, n=5):
        """Print and return the first n rows of the CSV file."""
        if self._data is None:
            self.read()
        head = self._data.head(n)
        print(head)
        return head

#### Exercise 2: Apply Pattern for Missing Values Handling

Objective:<br>
•	Learn the Strategy Pattern - one interface, many interchangeable behaviors.<br>
•	Practice inheritance and polymorphism.

In [8]:
from abc import ABC, abstractmethod

class MissingValueStrategy(ABC):
    """Abstract interface for all missing-value strategies."""

    @abstractmethod
    def handle(self, df):
        raise NotImplementedError("Subclasses must implement this method.")

class MissingStrategy(MissingValueStrategy):
    """Backward-compatible alias used in the initial notebook version."""
    def handle(self, df):
        return self.apply(df)
    def apply(self, df):
        raise NotImplementedError("Subclasses must implement this method.")

class DropMissing(MissingStrategy):
    def apply(self, df):
        """Drop rows with missing values."""
        return df.dropna().copy()

class FillMean(MissingStrategy):
    def apply(self, df):
        """Fill missing numeric values with the mean of each column."""
        numeric = df.select_dtypes(include=['number'])
        if numeric.empty:
            return df.copy()
        result = df.copy()
        for col in numeric.columns:
            mean_value = numeric[col].mean()
            result[col] = result[col].fillna(mean_value)
        return result

class FillMode(MissingStrategy):
    def apply(self, df):
        """Fill missing values with the mode of each column."""
        result = df.copy()
        for col in result.columns:
            mode_value = result[col].mode()
            if not mode_value.empty:
                result[col] = result[col].fillna(mode_value.iloc[0])
        return result

class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy
    def clean(self, df):
        """Clean the DataFrame using the selected missing value strategy."""
        return self.strategy.handle(df)

#### Exercise 3: Add Decorators for Logging and Timing

Objective:<br>
• Learn the Decorator Pattern to add reusable functionality like logging and timing.<br>
• Understand how decorators extend behavior without modifying the original code.


In [9]:
# Reuse the decorators defined in Exercise 1 and apply them to the cleaning workflow.
reader = CSVReader(r"sample_data.csv")
df = reader.read()
cleaned_df = DataCleaner(FillMean()).clean(df)
print(cleaned_df.head())

   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0


#### Exercise 4: Implement Factory Pattern for Data Transformations

Objective:<br>
• Practice the Factory Pattern for scalable creation of transformation objects.<br>
• Apply abstraction and composition in a data-processing pipeline.


In [10]:
from abc import ABC, abstractmethod

class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        raise NotImplementedError("Subclasses must implement this method.")

class NormalizeColumns(DataTransform):
    def apply(self, df):
        result = df.copy()
        result.columns = [str(col).strip().lower().replace(" ", "_") for col in result.columns]
        return result

class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates().copy()

class StandardizeText(DataTransform):
    def apply(self, df):
        result = df.copy()
        for col in result.select_dtypes(include=['object']).columns:
            result[col] = result[col].astype(str).str.strip().str.title()
        return result

class TransformFactory:
    @staticmethod
    def create(transform_name):
        factory = {
            'normalize_columns': NormalizeColumns,
            'remove_duplicates': RemoveDuplicates,
            'standardize_text': StandardizeText,
        }
        key = transform_name.strip().lower().replace(' ', '_')
        if key not in factory:
            raise ValueError(f"Unsupported transformation: {transform_name}")
        return factory[key]()

sample_df = pd.read_csv(r"sample_data.csv")
transformed = TransformFactory.create("normalize_columns").apply(sample_df)
print(transformed.columns)

Index(['id', 'name', 'age', 'height_cm', 'weight_kg', 'city', 'score'], dtype='object')


#### Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

Objective:<br>
• Combine all previous concepts into one cleaning pipeline.<br>
• Use the Template Method Pattern to define a repeatable workflow.


In [12]:
class DataPipeline(ABC):
    def __init__(self, file_path, output_path=None):
        self.file_path = file_path
        self.output_path = output_path or file_path.replace('.csv', '_cleaned.csv')
        self.data = None

    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self.data

    @abstractmethod
    def load(self):
        raise NotImplementedError("Subclasses must implement this method.")

    @abstractmethod
    def clean(self):
        raise NotImplementedError("Subclasses must implement this method.")

    @abstractmethod
    def transform(self):
        raise NotImplementedError("Subclasses must implement this method.")

    @abstractmethod
    def save(self):
        raise NotImplementedError("Subclasses must implement this method.")

In [13]:
class CSVDataPipeline(DataPipeline):
    def __init__(self, file_path, output_path=None, strategy=None, transform_name='normalize_columns'):
        super().__init__(file_path, output_path)
        self.reader = CSVReader(file_path)
        self.cleaner = DataCleaner(strategy or FillMean())
        self.transformer = TransformFactory.create(transform_name)

    def load(self):
        self.data = self.reader.read()

    def clean(self):
        self.data = self.cleaner.clean(self.data)

    def transform(self):
        self.data = self.transformer.apply(self.data)

    def save(self):
        self.data.to_csv(self.output_path, index=False)
        print(f"Saved cleaned data to: {self.output_path}")

pipeline = CSVDataPipeline(r"sample_data.csv")
result = pipeline.run()
print(result.head())

Saved cleaned data to: sample_data_cleaned.csv
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
